In [3]:
from transformers import pipeline


# 1. Muat (Load) Model dan Tokenizer menggunakan Pipeline
# Pipeline akan secara otomatis memilih tugas 'sentiment-analysis'
# dan menggunakan model 'siebert/sentiment-roberta-large-english'.
classifier = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english")

# 2. Daftar Teks Input (Contoh Teks Politik Bahasa Inggris)
political_texts = [
    "The new economic policy introduced by the administration is a disaster and will harm countless families.",
    "I believe the committee's decision to approve the infrastructure bill is a monumental step forward for our nation's progress.",
    "The recent budget proposal is neither good nor bad; it's just adequate for the current situation."
]

print("### Hasil Analisis Sentimen ###")
print("-" * 30)

# 3. Lakukan Prediksi Sentimen
results = classifier(political_texts)

# 4. Tampilkan Hasil
for text, result in zip(political_texts, results):
    label = result['label']
    score = result['score'] * 100 # Konversi ke persentase

    print(f"TEKS: '{text}'")
    print(f"SENTIMEN: **{label}**")
    print(f"KEPERCAYAAN: {score:.2f}%")
    print("-" * 30)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


### Hasil Analisis Sentimen ###
------------------------------
TEKS: 'The new economic policy introduced by the administration is a disaster and will harm countless families.'
SENTIMEN: **NEGATIVE**
KEPERCAYAAN: 99.94%
------------------------------
TEKS: 'I believe the committee's decision to approve the infrastructure bill is a monumental step forward for our nation's progress.'
SENTIMEN: **POSITIVE**
KEPERCAYAAN: 99.89%
------------------------------
TEKS: 'The recent budget proposal is neither good nor bad; it's just adequate for the current situation.'
SENTIMEN: **POSITIVE**
KEPERCAYAAN: 99.81%
------------------------------


In [31]:
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer


def check_mlflow_model_pipeline(model_uri: str):
    print("Model URI:", model_uri)

    # Load sebagai sklearn model, bukan pyfunc
    model = mlflow.sklearn.load_model(model_uri)

    print("Tipe model:", type(model))

    if isinstance(model, Pipeline):
        print("\nModel ini adalah sklearn Pipeline.")
        print("\nIsi pipeline:")

        for step_name, step_object in model.steps:
            print(f"- {step_name}: {type(step_object)}")

        print("\nNamed steps:")
        print(model.named_steps.keys())

        # Cek TextPreprocessor
        has_preprocessor = any(
            "preprocess" in step_name.lower()
            or step_object.__class__.__name__ == "TextPreprocessor"
            for step_name, step_object in model.steps
        )

        # Cek TF-IDF
        has_tfidf = any(
            isinstance(step_object, TfidfVectorizer)
            or "tfidf" in step_name.lower()
            for step_name, step_object in model.steps
        )

        # Cek classifier, biasanya step terakhir
        last_step_name, last_step_object = model.steps[-1]
        has_classifier = hasattr(last_step_object, "predict")

        print("\nHasil pengecekan:")
        print("Ada TextPreprocessor :", has_preprocessor)
        print("Ada TF-IDF           :", has_tfidf)
        print("Ada Classifier       :", has_classifier)
        print("Nama classifier      :", last_step_name)
        print("Tipe classifier      :", type(last_step_object))

        return model

    else:
        print("\nModel ini BUKAN sklearn Pipeline.")
        print("Kemungkinan yang tersimpan hanya classifier, atau model diload sebagai format lain.")
        return model

In [32]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
model_uri = "models:/sentiment_analysis_logistic_regression_optuna_best/Production"

In [33]:
check_mlflow_model_pipeline(model_uri)

Model URI: models:/sentiment_analysis_logistic_regression_optuna_best/Production


MlflowException: Registered Model with name=sentiment_analysis_logistic_regression_optuna_best not found